In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec, font_manager
from matplotlib.patches import Rectangle, FancyArrowPatch, FancyBboxPatch

In [ ]:
fontsize = 22
tick_step = 2  # show every 2nd tick
n_patches = 12
d_v = 8  # dimension of value (attention output dimension)

# Manual sans-serif fonts
sans_font = font_manager.FontProperties(family='DejaVu Sans', size=fontsize)
sans_font_bold = font_manager.FontProperties(family='DejaVu Sans', size=fontsize, weight='bold')


# Color palette
color_palette = ['#C0D6CA', '#78ACA8', '#2D6B8F', '#235796', 
                 '#E7C4C0', '#E3A39A', '#CA6F6A', '#7B3841',
                 '#D5BC67', '#20425B', '#E77A5B', '#9C9DB2']

tick_positions_patch = np.arange(0, n_patches, tick_step)
tick_labels_patch = np.arange(1, n_patches+1, tick_step)
tick_positions_d = np.arange(0, d_v, 2)
tick_labels_d = np.arange(1, d_v+1, 2)

# -----------------------------
# FIGURE SETUP
# -----------------------------
fig = plt.figure(figsize=(14, 7))
gs = gridspec.GridSpec(7, 8, figure=fig, hspace=0.25, wspace=0.35, 
                       width_ratios=[1, 1, 0.7, 1, 1, 1.5, 1, 1])

# -----------------------------
# WIND SPEED DATA
# -----------------------------
np.random.seed(42)
n_timesteps = 120
time = np.arange(n_timesteps)

def generate_wind_data(base_speed, gust_times, phase_shift):
    data = base_speed + 3*np.sin(time*0.2 + phase_shift) + np.random.randn(n_timesteps)*0.5
    for gust_time in gust_times:
        if gust_time < n_timesteps:
            data[gust_time:min(gust_time+3, n_timesteps)] += np.array([8,10,6])[:min(3, n_timesteps-gust_time)]
    return data

stations = [
    ('Series A', generate_wind_data(12, [20,60], 0)),
    ('Series B', generate_wind_data(14, [22,62], 0.5)),
    ('Series C', generate_wind_data(10, [24,64], 1.0))
]

# -----------------------------
# LEFT: STACKED TIME SERIES
# -----------------------------
iis = [0,3,5]
for i, (name, data) in enumerate(stations):
    ax = fig.add_subplot(gs[iis[i]:iis[i]+2, 0:2])
    ax.plot(time, data, color='black', linewidth=1.5)
    ax.set_ylabel(name, fontproperties=sans_font)
    ax.set_xlim(0, n_timesteps-1)
    ax.set_ylim(5,30)
    ax.tick_params(axis='y', labelsize=fontsize)
    ax.tick_params(axis='x', labelsize=fontsize)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linewidth=0.5)
    if i < 2:
        ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    else:
        ax.set_xlabel('Time', fontproperties=sans_font)

# -----------------------------
# ARROWS FROM TIME SERIES
# -----------------------------
def add_arrow(gs_row, gs_col, y_pos, start, end, n_span=1, color='black'):
    ax_arrow = fig.add_subplot(gs[gs_row, gs_col:gs_col+n_span])
    ax_arrow.axis('off')
    arrow = FancyArrowPatch((start, y_pos), (end, y_pos), arrowstyle='->', mutation_scale=20, linewidth=1.5, color=color)
    ax_arrow.add_patch(arrow)
    return ax_arrow

add_arrow(1, 1, 0.3, 0.49, 0.75, 2)  # Station A -> local
add_arrow(4, 1, 0.4, 0.49, 0.75, 2)  # Station B -> global
add_arrow(6, 1, 0.4, 0.49, 0.75, 2)  # Station C -> global
add_arrow(2, 4, 0.6, 0, 0.525, 2)
add_arrow(4, 4, 0.2, 0, 0.525, 2)
add_arrow(3, 5, 0.4, 0.42, 0.73)


# -----------------------------
# FUNCTION TO DRAW ATTENTION OUTPUT USING RECTANGLES
# -----------------------------
def draw_attention_output(ax, matrix, cmap='Blues', title='Attention Output'):
    """Draw attention output matrix (n_patches × d_v)"""
    n_rows, n_cols = matrix.shape
    ax.set_xlim(-0.5, n_cols-0.5)
    ax.set_ylim(-0.5, n_rows-0.5)
    ax.set_aspect('auto')
    ax.set_xticks(tick_positions_d)
    ax.set_yticks(tick_positions_patch)
    ax.set_xticklabels(tick_labels_d, fontproperties=sans_font)
    ax.set_yticklabels(tick_labels_patch, fontproperties=sans_font)
    ax.set_xlabel('$d$', fontproperties=sans_font)
    ax.set_ylabel('Patch Index', fontproperties=sans_font)
    ax.set_title(title, fontproperties=sans_font_bold, color='black', pad=10)
    ax.grid(True, which='major', color='white', linewidth=0.5)

    if title=='Local Attention':
        ax.set_xlabel('', fontproperties=sans_font)

    # Normalize values for coloring
    mat_norm = (matrix - matrix.min()) / (matrix.max() - matrix.min())
    for i in range(n_rows):
        for j in range(n_cols):
            rect = Rectangle((j-0.5, i-0.5), 1, 1, 
                             facecolor=plt.cm.Blues(mat_norm[i,j]), edgecolor='white')
            ax.add_patch(rect)

# -----------------------------
# LOCAL ATTENTION OUTPUT (n_patches × d_v)
# -----------------------------
ax_local = fig.add_subplot(gs[0:3, 3:5])
# Local attention: high variation across patches (different temporal contexts)
local_output = np.zeros((n_patches, d_v))
for i in range(n_patches):
    for j in range(d_v):
        # Each patch has distinct features based on its temporal position
        local_output[i, j] = 0.4 + 0.4*np.sin(i*0.5 + j*0.8) + np.random.rand()*0.15
local_output = np.flipud(local_output)
draw_attention_output(ax_local, local_output, title='Local Attention')

# -----------------------------
# GLOBAL ATTENTION OUTPUT (n_patches × d_v)
# -----------------------------
ax_global = fig.add_subplot(gs[4:7, 3:5])
# Global attention: more uniform across patches (shared cross-channel information)
global_output = np.zeros((n_patches, d_v))
base_pattern = 0.5 + 0.3*np.sin(np.arange(d_v)*0.6)  # Shared pattern across patches
for i in range(n_patches):
    for j in range(d_v):
        # Similar patterns across patches with slight variation
        global_output[i, j] = base_pattern[j] + np.random.rand()*0.1
global_output = np.flipud(global_output)
draw_attention_output(ax_global, global_output, title='Global Attention')

# -----------------------------
# OUTPUT ATTENTION (mixed)
# -----------------------------
ax_output = fig.add_subplot(gs[2:5, 6:8])
# Mixed output: combines local variation with global structure
output_matrix = 0.5*local_output + 0.5*global_output
output_matrix = (output_matrix - output_matrix.min()) / (output_matrix.max() - output_matrix.min())
draw_attention_output(ax_output, output_matrix, title='Output Attention')

# -----------------------------
# MIXING GATE
# -----------------------------
ax_gate = fig.add_subplot(gs[1:6, 5])
ax_gate.axis('off')
ax_gate.set_xlim(0,1)
ax_gate.set_ylim(0,1)
gate_box = FancyBboxPatch((0.1,0.2), 0.3, 0.58, boxstyle="round,pad=0.03",
                          edgecolor=color_palette[2], facecolor=color_palette[2], alpha=0.7, linewidth=1.5)
ax_gate.add_patch(gate_box)
ax_gate.text(0.27, 0.5, 'Mixing Gate', ha='center', va='center', rotation=90,
             fontproperties=sans_font_bold, color='white')

plt.tight_layout()
plt.savefig('./summary_figure.pdf')
